# Кластеризация и классификация данных: от поиска групп к прогнозу

**Кейс:** анализ обращений службы поддержки.

За занятие мы последовательно ответим на два вопроса аналитика:

1. **Кластеризация:** какие естественные группы обращений можно обнаружить без готовых меток?
2. **Классификация:** можно ли по информации, доступной в начале работы, спрогнозировать риск нарушения SLA?

Главный принцип: модель — не конечный результат. После расчёта нужно проверить качество и сформулировать содержательный вывод.

## Как работать с notebook

Выполняйте ячейки **сверху вниз**. Код уже работоспособен. После основных блоков есть аналитические задания: сначала получите эталонный результат, затем меняйте параметры или формулируйте выводы своими словами.

In [ ]:
# Импортируем стандартные библиотеки для работы с файлами и системной информацией.
from pathlib import Path
import sys

# Импортируем библиотеки для работы с таблицами и числовыми массивами.
import numpy as np
import pandas as pd

# Импортируем библиотеку для построения графиков.
import matplotlib.pyplot as plt

# Импортируем инструменты для кластеризации и подготовки числовых признаков.
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import silhouette_score

# Импортируем инструменты для классификации и честной оценки на тестовой выборке.
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Импортируем метрики качества классификации и визуализацию матрицы ошибок.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

# Настраиваем pandas так, чтобы таблицы в notebook отображались удобнее.
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

# Печатаем версии Python и основных библиотек для диагностики окружения.
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('numpy:', np.__version__)

In [ ]:
# Определяем текущую рабочую папку.
cwd = Path.cwd()

# Формируем список возможных корней проекта.
# Первый вариант подходит при запуске Jupyter из корня проекта,
# второй — при запуске notebook из папки notebooks/.
project_candidates = [cwd, cwd.parent]

# Ищем структуру data/raw/support_tickets.csv.
project_root = None
for candidate in project_candidates:
    if (candidate / 'data' / 'raw' / 'support_tickets.csv').exists():
        project_root = candidate
        break

# Если проектной структуры нет, проверяем вариант Google Colab:
# CSV может быть загружен рядом с notebook в текущую папку.
if project_root is None and (cwd / 'support_tickets.csv').exists():
    project_root = cwd

# Если файл так и не найден, останавливаемся с понятным сообщением.
if project_root is None:
    raise FileNotFoundError(
        'Не найден support_tickets.csv. Поместите файл в data/raw/ '
        'или загрузите его рядом с notebook в Google Colab.'
    )

# Выбираем фактический путь к исходному CSV.
project_data_path = project_root / 'data' / 'raw' / 'support_tickets.csv'
data_path = project_data_path if project_data_path.exists() else project_root / 'support_tickets.csv'

# Создаём папки для результатов и графиков, если их ещё нет.
output_dir = project_root / 'outputs'
charts_dir = output_dir / 'charts'
output_dir.mkdir(parents=True, exist_ok=True)
charts_dir.mkdir(parents=True, exist_ok=True)

# Показываем найденные пути — это помогает быстро диагностировать проблемы с файлами.
print('Рабочая папка:', cwd)
print('Корень проекта:', project_root)
print('Данные:', data_path)
print('Результаты:', output_dir)

## Шаг 1. Проверяем исходные данные

Перед любой моделью аналитик проверяет, что данные загрузились ожидаемым образом. В этом кейсе мы не занимаемся отдельной большой очисткой: данные подготовлены для изучения методов кластеризации и классификации.

In [ ]:
# Загружаем CSV и сразу преобразуем created_at в тип datetime.
tickets = pd.read_csv(data_path, parse_dates=['created_at'])

# Показываем первые строки, чтобы визуально проверить структуру таблицы.
display(tickets.head())

# Проверяем размер датасета.
print('Размер таблицы:', tickets.shape)

# Проверяем количество пропусков по столбцам.
missing = tickets.isna().sum()
print('Всего пропусков:', int(missing.sum()))
display(missing.to_frame('missing_count').T)

# Проверяем уникальность идентификатора обращения.
duplicate_ids = tickets['ticket_id'].duplicated().sum()
print('Дубликаты ticket_id:', int(duplicate_ids))

# Проверяем баланс целевого класса заранее, чтобы позже правильно читать accuracy.
class_share = tickets['sla_breached'].value_counts(normalize=True).sort_index()
print('Доли классов sla_breached:')
display(class_share.rename('share').to_frame())

**Контрольный вопрос:** почему долю класса `sla_breached=1` полезно посмотреть ещё до обучения классификатора? Запишите ответ своими словами.

## Шаг 2. Ищем группы визуально

Начинаем максимально просто: посмотрим только на время первого ответа и время решения. Пока никакого алгоритма нет.

In [ ]:
# Строим первый простой график без модели.
# Каждая точка — одно завершённое обращение.
plt.figure(figsize=(9, 5))
plt.scatter(
    tickets['first_response_minutes'],
    tickets['resolution_minutes'],
    alpha=0.35,
    s=18,
)
plt.xlabel('Время до первого ответа, мин')
plt.ylabel('Время решения, мин')
plt.title('Обращения до кластеризации: видны ли группы?')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(charts_dir / '01_initial_scatter.png', dpi=140, bbox_inches='tight')
plt.show()

**Задание:** опишите, видите ли вы на графике зоны с разной плотностью точек. Не пытайтесь пока давать им окончательные названия.

## Шаг 3. Первый K-Means на двух признаках

Теперь алгоритм автоматически присвоит каждой точке технический номер кластера.

In [ ]:
# Берём только два признака, чтобы первый пример K-Means можно было увидеть на плоскости.
simple_features = ['first_response_minutes', 'resolution_minutes']
X_simple = tickets[simple_features].copy()

# Масштабируем оба признака, потому что они имеют разные диапазоны значений.
simple_scaler = StandardScaler()
X_simple_scaled = simple_scaler.fit_transform(X_simple)

# Создаём модель K-Means с тремя кластерами.
# random_state фиксирует воспроизводимость, n_init задаёт несколько стартовых инициализаций.
simple_kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)

# Обучаем модель и получаем номер кластера для каждого обращения.
simple_cluster = simple_kmeans_model.fit_predict(X_simple_scaled)

# Строим тот же scatter plot, но теперь раскрашиваем точки по найденному кластеру.
plt.figure(figsize=(9, 5))
scatter = plt.scatter(
    tickets['first_response_minutes'],
    tickets['resolution_minutes'],
    c=simple_cluster,
    alpha=0.45,
    s=18,
)
plt.xlabel('Время до первого ответа, мин')
plt.ylabel('Время решения, мин')
plt.title('Первый K-Means на двух признаках')
plt.legend(*scatter.legend_elements(), title='Кластер')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(charts_dir / '02_simple_kmeans.png', dpi=140, bbox_inches='tight')
plt.show()

**Мини-эксперимент:** после первого успешного запуска измените `n_clusters=3` на `2` или `4`, выполните ячейку ещё раз и сравните картину. После эксперимента верните `3`, чтобы дальнейшие результаты совпадали с контрольными.

## Шаг 4. Переходим к нескольким признакам и масштабированию

В реальной аналитике одна пара признаков редко описывает объект полностью. Добавим интенсивность коммуникации, повторные открытия, очередь и историю обращений.

In [ ]:
# Для более содержательной кластеризации используем шесть характеристик завершённого обращения.
cluster_features = [
    'first_response_minutes',
    'resolution_minutes',
    'messages_count',
    'reopen_count',
    'backlog_at_creation',
    'previous_tickets_90d',
]

# Создаём матрицу признаков для кластеризации.
X_cluster = tickets[cluster_features].copy()

# Масштабируем признаки, чтобы единицы измерения не определяли расстояние в K-Means.
cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)

# Проверяем средние и стандартные отклонения после масштабирования.
scaled_check = pd.DataFrame(X_cluster_scaled, columns=cluster_features).agg(['mean', 'std']).round(2)
display(scaled_check)

**Контрольный вопрос:** почему `resolution_minutes` с сотнями минут может чрезмерно влиять на расстояния по сравнению с `reopen_count`, где значения обычно единичные?

## Шаг 5. Выбираем количество кластеров

Сравним несколько значений `k`. Здесь метрики помогают аналитику, но не заменяют содержательную проверку профилей.

In [ ]:
# Проверяем несколько вариантов количества кластеров.
k_values = range(2, 7)
k_results = []

for k in k_values:
    # Для каждого k создаём и обучаем отдельную модель.
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_cluster_scaled)

    # Inertia показывает сумму квадратов расстояний до центров кластеров.
    inertia = model.inertia_

    # Silhouette score оценивает одновременно компактность своего кластера и отделённость от соседних.
    silhouette = silhouette_score(X_cluster_scaled, labels)

    # Сохраняем показатели для последующего сравнения.
    k_results.append({'k': k, 'inertia': inertia, 'silhouette': silhouette})

# Собираем результаты в таблицу.
k_quality = pd.DataFrame(k_results)
display(k_quality.round(3))

# Отдельно строим график inertia.
plt.figure(figsize=(8, 4.5))
plt.plot(k_quality['k'], k_quality['inertia'], marker='o')
plt.xlabel('Количество кластеров k')
plt.ylabel('Inertia')
plt.title('Метод локтя: как меняется внутрикластерный разброс')
plt.xticks(list(k_values))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(charts_dir / '03_k_inertia.png', dpi=140, bbox_inches='tight')
plt.show()

# Отдельно строим график silhouette score.
plt.figure(figsize=(8, 4.5))
plt.plot(k_quality['k'], k_quality['silhouette'], marker='o')
plt.xlabel('Количество кластеров k')
plt.ylabel('Silhouette score')
plt.title('Silhouette score для разных значений k')
plt.xticks(list(k_values))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(charts_dir / '04_k_silhouette.png', dpi=140, bbox_inches='tight')
plt.show()

**Задание:** найдите `k` с максимальным silhouette score. Сравните его с формой графика inertia. Почему одного числа всё равно недостаточно для окончательной интерпретации?

## Шаг 6. Строим итоговые кластеры и профилируем их

Самая важная часть для аналитика — не номер кластера, а понимание, чем группы отличаются в исходных единицах.

In [ ]:
# Для данного учебного набора выбираем k=3:
# это значение даёт хорошо разделённые и содержательно интерпретируемые группы.
final_k = 3

# Обучаем итоговую модель на масштабированных признаках.
kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=10)
tickets['cluster'] = kmeans.fit_predict(X_cluster_scaled)

# Считаем профиль каждого кластера в ИСХОДНЫХ единицах измерения.
# Именно эта таблица нужна аналитику для интерпретации групп.
cluster_profile = (
    tickets.groupby('cluster')[cluster_features]
    .mean()
    .round(1)
)

# Добавляем количество обращений в каждом кластере.
cluster_profile.insert(0, 'tickets_count', tickets.groupby('cluster').size())

display(cluster_profile)

# Сохраняем данные с кластерами и таблицу профилей.
tickets.to_csv(output_dir / 'support_tickets_with_clusters.csv', index=False, encoding='utf-8')
cluster_profile.to_csv(output_dir / 'cluster_profile.csv', encoding='utf-8')

print('Сохранено:', output_dir / 'support_tickets_with_clusters.csv')
print('Сохранено:', output_dir / 'cluster_profile.csv')

In [ ]:
# Визуализируем итоговые кластеры на двух понятных признаках.
plt.figure(figsize=(9, 5))
scatter = plt.scatter(
    tickets['first_response_minutes'],
    tickets['resolution_minutes'],
    c=tickets['cluster'],
    alpha=0.45,
    s=18,
)
plt.xlabel('Время до первого ответа, мин')
plt.ylabel('Время решения, мин')
plt.title('Итоговые кластеры обращений')
plt.legend(*scatter.legend_elements(), title='Кластер')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(charts_dir / '05_final_clusters.png', dpi=140, bbox_inches='tight')
plt.show()

**Практическое задание:** дайте каждому кластеру содержательное рабочее название. Используйте одновременно несколько столбцов профиля, а не один показатель. Запишите 1–2 предложения по каждому кластеру.

# Часть 2. Классификация риска нарушения SLA

Теперь постановка меняется. В исторических данных уже есть правильный класс `sla_breached`. Значит можно обучить модель предсказывать этот класс для новых обращений.

## Шаг 7. Не допускаем target leakage

Хотим оценивать риск **в момент создания обращения**. Поэтому нельзя подавать модели признаки, которые станут известны только после обработки: `resolution_minutes`, `messages_count`, `reopen_count`, `first_response_minutes`.

Это принципиально: высокая метрика модели с информацией из будущего не делает такую модель полезной.

In [ ]:
# Создаём производные признаки, доступные уже в момент создания обращения.
model_data = tickets.copy()
model_data['created_hour'] = model_data['created_at'].dt.hour
model_data['is_weekend'] = (model_data['created_at'].dt.dayofweek >= 5).astype(int)

# Числовые признаки доступны до завершения обращения.
numeric_features = [
    'previous_tickets_90d',
    'backlog_at_creation',
    'agent_experience_months',
    'created_hour',
    'is_weekend',
]

# Категориальные признаки также известны в начале работы.
categorical_features = [
    'channel',
    'category',
    'priority',
    'customer_segment',
]

# Собираем признаки X и целевую переменную y.
# Обратите внимание: resolution_minutes, messages_count и reopen_count сюда НЕ входят,
# потому что это информация, появляющаяся после начала обработки обращения.
X = model_data[numeric_features + categorical_features].copy()
y = model_data['sla_breached'].copy()

# Делим данные на обучающую и тестовую выборки.
# stratify сохраняет примерно одинаковую долю классов в train и test.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Доля нарушений в train:', round(y_train.mean(), 3))
print('Доля нарушений в test:', round(y_test.mean(), 3))

## Шаг 8. Сначала baseline

Baseline показывает, насколько сложно превзойти самое простое правило. Без baseline легко принять красивую метрику за реальную ценность.

In [ ]:
# Создаём простейшую baseline-модель: она всегда выбирает самый частый класс.
baseline_model = DummyClassifier(strategy='most_frequent')

# Обучаем baseline только на train.
baseline_model.fit(X_train, y_train)

# Получаем прогноз на независимой тестовой выборке.
baseline_pred = baseline_model.predict(X_test)

# Печатаем accuracy baseline как минимальную точку сравнения.
print('Baseline accuracy:', round(accuracy_score(y_test, baseline_pred), 3))

## Шаг 9. Обучаем две модели

Сравним Logistic Regression и Decision Tree. Для категориальных признаков используем one-hot encoding, для числовых — масштабирование. Всё preprocessing находится внутри `Pipeline`, поэтому обучается только на train.

In [ ]:
# Функция создаёт НОВЫЙ preprocessing для каждой модели.
# Числовые признаки масштабируются, категориальные кодируются one-hot способом.
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ]
    )

# Logistic Regression используем как простую и понятную базовую обучаемую модель.
logistic_model = Pipeline(
    steps=[
        ('preprocess', make_preprocessor()),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
    ]
)

# Decision Tree используем как второй алгоритм для сравнения подходов.
tree_model = Pipeline(
    steps=[
        ('preprocess', make_preprocessor()),
        ('model', DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=20,
            class_weight='balanced',
            random_state=42,
        )),
    ]
)

# Обучаем обе модели только на обучающей выборке.
logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

# Получаем прогнозы для test.
logistic_pred = logistic_model.predict(X_test)
tree_pred = tree_model.predict(X_test)

# Получаем вероятность класса 1 для Logistic Regression.
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

print('Модели обучены. Размер тестового прогноза:', len(logistic_pred))

## Шаг 10. Сравниваем модели по нескольким метрикам

В задаче SLA особенно важно не пропускать реальные нарушения. Поэтому смотрим не только accuracy, но и recall, precision и F1.

In [ ]:
# Создаём функцию, чтобы считать одинаковый набор метрик для всех моделей.
def calculate_metrics(y_true, y_pred, model_name):
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }

# Рассчитываем метрики baseline и двух обученных моделей.
metrics = pd.DataFrame([
    calculate_metrics(y_test, baseline_pred, 'Baseline: most_frequent'),
    calculate_metrics(y_test, logistic_pred, 'Logistic Regression'),
    calculate_metrics(y_test, tree_pred, 'Decision Tree'),
])

# Округляем только отображение, исходные числа сохраняем с полной точностью.
display(metrics.round(3))

# Сохраняем таблицу метрик для следующего занятия по интерпретации результатов.
metrics.to_csv(output_dir / 'classification_metrics.csv', index=False, encoding='utf-8')
print('Сохранено:', output_dir / 'classification_metrics.csv')

In [ ]:
# Печатаем подробный classification report для Logistic Regression.
print('Logistic Regression')
print(classification_report(y_test, logistic_pred, digits=3))

# Печатаем подробный classification report для Decision Tree.
print('Decision Tree')
print(classification_report(y_test, tree_pred, digits=3))

**Задание:** какая модель лучше обнаруживает реальные нарушения SLA по `recall`? Какая модель имеет лучший `F1`? Совпадает ли выбор модели по этим двум критериям?

## Шаг 11. Разбираем типы ошибок

In [ ]:
# Строим confusion matrix для Logistic Regression.
ConfusionMatrixDisplay.from_predictions(y_test, logistic_pred)
plt.title('Confusion matrix: Logistic Regression')
plt.tight_layout()
plt.savefig(charts_dir / '06_confusion_logistic.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# Строим confusion matrix для Decision Tree.
ConfusionMatrixDisplay.from_predictions(y_test, tree_pred)
plt.title('Confusion matrix: Decision Tree')
plt.tight_layout()
plt.savefig(charts_dir / '07_confusion_tree.png', dpi=140, bbox_inches='tight')
plt.show()

**Главный аналитический вопрос:** какая ошибка для службы поддержки опаснее — False Positive или False Negative? Обоснуйте ответ через рабочую ситуацию, а не через определение метрики.

## Шаг 12. Сохраняем прогнозы

In [ ]:
# Собираем таблицу прогнозов именно для объектов из тестовой выборки.
predictions = model_data.loc[X_test.index, ['ticket_id', 'priority', 'category', 'sla_breached']].copy()

# Добавляем прогнозы обеих моделей и вероятность нарушения SLA от Logistic Regression.
predictions['logistic_prediction'] = logistic_pred
predictions['logistic_probability'] = np.round(logistic_prob, 4)
predictions['tree_prediction'] = tree_pred

# Сортируем по вероятности риска, чтобы аналитик мог увидеть самые рискованные обращения.
predictions = predictions.sort_values('logistic_probability', ascending=False)

# Показываем верхние строки.
display(predictions.head(10))

# Сохраняем результат для дальнейшей интерпретации и представления.
predictions.to_csv(output_dir / 'classification_predictions.csv', index=False, encoding='utf-8')
print('Сохранено:', output_dir / 'classification_predictions.csv')

## Финальная самопроверка

In [ ]:
# Проверяем наличие всех основных файлов, которые должен создать notebook.
expected_outputs = [
    output_dir / 'support_tickets_with_clusters.csv',
    output_dir / 'cluster_profile.csv',
    output_dir / 'classification_metrics.csv',
    output_dir / 'classification_predictions.csv',
]

for path in expected_outputs:
    print(('OK  ' if path.exists() else 'НЕТ '), path.name)

## Итог

После этого notebook вы должны уметь объяснить разницу:

- **кластеризация** ищет структуру без готовых классов;
- **классификация** учится воспроизводить известный класс по признакам;
- качество обоих подходов требует проверки;
- технический результат модели должен быть переведён в аналитический вывод.